# Getting Started with Strands Agents

This workshop introduces Strands Agents, a powerful framework for building AI agents with tool integration capabilities. You'll learn the fundamentals of creating agents, using built-in tools, and developing custom tools.

## Overview

In this lab, you will:
- Understand the core concepts of Strands Agents
- Create your first AI agent with built-in tools
- Build custom tools for specific use cases
- Explore conversation history and agent memory
- Learn best practices for agent development

## Prerequisites

Before starting this lab, ensure you have:
- AWS credentials configured (IAM role or environment variables)
- Required Python packages installed
- Nova Pro model ID based on AWS region

If you're not running in an environment with an IAM role assumed, set your AWS credentials as environment variables:

In [1]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

Install required packages for Strands Agents:

In [2]:
#%pip install -q strands-agents strands-agents-tools rich

Setup Nova Pro model ID based on AWS region:

⚠️ **Important**: If you haven't enabled Nova Pro model access yet, go to the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/home#/modelaccess) to enable it.

In [3]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Nova Pro Model ID: us.amazon.nova-pro-v1:0


## What are Strands Agents?

Strands Agents is a Python framework that simplifies the creation of AI agents with tool integration capabilities. Key features include:

- **Simple Agent Creation**: Easy-to-use API for creating AI agents with minimal code
- **Built-in Tools**: Pre-built tools like calculators, web search, and more
- **Custom Tool Support**: Create your own tools with simple Python functions
- **Conversation Memory**: Automatic conversation history management
- **Model Flexibility**: Support for various language models including models in Amazon Bedrock and OpenAI

Strands Agents provides a foundation for building sophisticated AI applications that can interact with external systems and perform complex tasks.

## Creating Your First Strands Agent

Let's start by creating a simple agent with a built-in calculator tool. This demonstrates the basic structure of Strands Agents and how tools are integrated:

In [4]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

# Create your first agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a helpful assistant that provides concise responses.",
    tools=[calculator],
)

agent("What is the area of a circle with radius 8.26cm?")

<thinking> To find the area of a circle, I need to use the formula A = πr², where r is the radius. The radius given is 8.26cm. I will use the 'evaluate' mode of the calculator tool to compute the area. </thinking>

Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ pi * 8.26**2        │                                                                            │
│  │ Result    │ 214.3433269318      │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The area of the circle with radius 8.26cm is approximately 214.34 cm².

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The area of the circle with radius 8.26cm is approximately 214.34 cm².'}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_Oz9aYztRTKMTMEUoGIcRUm', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': 'pi * 8.26**2'}}, call_count=1, success_count=1, error_count=0, total_time=0.014207124710083008)}, cycle_durations=[0.6608290672302246], traces=[<strands.telemetry.metrics.Trace object at 0xffff4896b6d0>, <strands.telemetry.metrics.Trace object at 0xffff49f80610>], accumulated_usage={'inputTokens': 3704, 'outputTokens': 112, 'totalTokens': 3816}, accumulated_metrics={'latencyMs': 2115}), state={})

## Building Custom Tools

One of the powerful features of Strands Agents is the ability to create custom tools. Let's create a weather tool and combine it with the built-in calculator:

In [5]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

# Create a custom weather tool for demonstration
@tool
def weather(city: str) -> str:
    """Get weather information for a city
    Args:
        city: City or location name
    """
    return f"Weather for {city}: Sunny, 35°C" # dummy result for demo purpose

# Create your first agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a helpful assistant that provides concise responses.",
    tools=[weather, calculator],
)

agent("How is the weather in HK? Return temperature in Fahrenheit")

<thinking> I need to get the weather information for Hong Kong and convert the temperature from Celsius to Fahrenheit. </thinking>

Tool #1: weather
<thinking> I have the temperature in Celsius, which is 35°C. I need to convert this to Fahrenheit using the formula F = (C * 9/5) + 32. </thinking> 
Tool #2: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 35 * 9/5 + 32       │                                                                            │
│  │ Result    │ 95                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The temperature in Hong Kong is currently 95°F.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The temperature in Hong Kong is currently 95°F.'}]}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'weather': ToolMetrics(tool={'toolUseId': 'tooluse_FdImeaCgjHU4bipYSbBA3n', 'name': 'weather', 'input': {'city': 'HK'}}, call_count=1, success_count=1, error_count=0, total_time=0.0003910064697265625), 'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_2IRfNSKsZiFCV1e6Jh0K5v', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': '35 * 9/5 + 32'}}, call_count=1, success_count=1, error_count=0, total_time=0.006810665130615234)}, cycle_durations=[0.5062038898468018], traces=[<strands.telemetry.metrics.Trace object at 0xffff60381bd0>, <strands.telemetry.metrics.Trace object at 0xffff48184990>, <strands.telemetry.metrics.Trace object at 0xffff4819dbd0>], accumulated_usage={'inputTokens': 5767, 'outputTokens': 128, 'totalTokens': 5895}, accumulated_metrics={'latencyMs': 2089}), state=

## Understanding Agent Execution Flow

Let's examine how Strands Agents process requests with an agent loop. This helps understand the internal workings of the agent framework:

In [6]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 3

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                        ┃ Tool Name  ┃ Tool Input                  ┃ Tool Result                ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ How is the weather in HK?   │            │                             │                            │
│           │ Return temperature in       │            │                             │                            │
│           │ Fahrenheit                  │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ <thinking> I need to get    │ weather    │ {                           │                            │
│           │ the weather information for │            │   "city": "HK"              │                            │
│           │ Hong Kong and convert the   │            │ }                           │                            │
│           │ temperature from Celsius to │            │                             │                            │
│           │ Fahrenheit. </thinking>     │            │                             │                            │
│           │                             │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │                             │            │                             │ {                          │
│           │                             │            │                             │   "text": "Weather for HK: │
│           │                             │            │                             │ Sunny, 35\u00b0C"          │
│           │                             │            │                             │ }                          │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ <thinking> I have the       │ calculator │ {                           │                            │
│           │ temperature in Celsius,     │            │   "mode": "evaluate",       │                            │
│           │ which is 35°C. I need to    │            │   "expression": "35 * 9/5 + │                            │
│           │ convert this to Fahrenheit  │            │ 32"                         │                            │
│           │ using the formula F = (C *  │            │ }                           │                            │
│           │ 9/5) + 32. </thinking>      │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │                             │            │                             │ {                          │
│           │                             │            │                             │   "text": "Result: 95"     │
│           │                             │            │                             │ }                          │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ The temperature in Hong     │            │                             │                            │
│           │ Kong is currently 95°F.     │            │                             │                            │
└───────────┴─────────────────────────────┴────────────┴─────────────────────────────┴────────────────────────────┘

## Agent Loop

The agent loop is a core concept in the Strands Agents SDK that enables intelligent, autonomous behavior through a cycle of reasoning, tool use, and response generation. 

![strands-agents-agent-loop](images/strands-agents-agent-loop.png)

At its core, the agent loop follows these steps:

1. Receives user input and contextual information
2. Processes the input using a language model (LLM)
3. Decides whether to use tools to gather information or perform actions
4. Executes tools and receives results
5. Continues reasoning with the new information
6. Produces a final response or iterates again through the loop
   
This cycle may repeat multiple times within a single user interaction, allowing the agent to perform complex, multi-step reasoning and autonomous behavior.

Reference: [Strands Agents - Agent Loop](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/agent-loop/)


## Strands Agents Conversation Management

Strands Agents includes built-in conversation management with a default `SlidingWindowConversationManager` strategy. This system handles context management, memory optimization, and conversation flow to ensure agents can maintain coherent long-term interactions while respecting model context limits. This automatically maintains conversation context within sessions.

**Reference:** [Strands Agents - Conversation Management](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/conversation-management/)

In [7]:
agent("What did we talk about?")

We discussed the weather in Hong Kong. I provided the current temperature in Fahrenheit after converting it from Celsius.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'We discussed the weather in Hong Kong. I provided the current temperature in Fahrenheit after converting it from Celsius.'}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'weather': ToolMetrics(tool={'toolUseId': 'tooluse_FdImeaCgjHU4bipYSbBA3n', 'name': 'weather', 'input': {'city': 'HK'}}, call_count=1, success_count=1, error_count=0, total_time=0.0003910064697265625), 'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_2IRfNSKsZiFCV1e6Jh0K5v', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': '35 * 9/5 + 32'}}, call_count=1, success_count=1, error_count=0, total_time=0.006810665130615234)}, cycle_durations=[0.5062038898468018, 0.8773589134216309], traces=[<strands.telemetry.metrics.Trace object at 0xffff60381bd0>, <strands.telemetry.metrics.Trace object at 0xffff48184990>, <strands.telemetry.metrics.Trace object at 0xffff4819dbd0>, <strands.telemetry.metrics.Trace objec

Let's examine how Strands Agents manage conversation history. This helps understand the internal workings of the agent framework:

In [8]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 4

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                        ┃ Tool Name  ┃ Tool Input                  ┃ Tool Result                ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ How is the weather in HK?   │            │                             │                            │
│           │ Return temperature in       │            │                             │                            │
│           │ Fahrenheit                  │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ <thinking> I need to get    │ weather    │ {                           │                            │
│           │ the weather information for │            │   "city": "HK"              │                            │
│           │ Hong Kong and convert the   │            │ }                           │                            │
│           │ temperature from Celsius to │            │                             │                            │
│           │ Fahrenheit. </thinking>     │            │                             │                            │
│           │                             │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │                             │            │                             │ {                          │
│           │                             │            │                             │   "text": "Weather for HK: │
│           │                             │            │                             │ Sunny, 35\u00b0C"          │
│           │                             │            │                             │ }                          │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ <thinking> I have the       │ calculator │ {                           │                            │
│           │ temperature in Celsius,     │            │   "mode": "evaluate",       │                            │
│           │ which is 35°C. I need to    │            │   "expression": "35 * 9/5 + │                            │
│           │ convert this to Fahrenheit  │            │ 32"                         │                            │
│           │ using the formula F = (C *  │            │ }                           │                            │
│           │ 9/5) + 32. </thinking>      │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │                             │            │                             │ {                          │
│           │                             │            │                             │   "text": "Result: 95"     │
│           │                             │            │                             │ }                          │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ The temperature in Hong     │            │                             │                            │
│           │ Kong is currently 95°F.     │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │ What did we talk about?     │            │

### Examine Strands Agents Conversation Management in a New Session

Strands Agents Conversation Management maintains the conversation history by accumulating the user and agent messages in input prompt within a session. The conversation history is stored in memory and is volatile when the session is end. Let's simulate a new session by initialising a new agent and test the Strands Agents Conversation Management behavior.

In [9]:
# Initialise a new agent for a new session
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a helpful assistant that provides concise responses.",
    tools=[weather, calculator],
)

agent("What did we talk about?")

<thinking> There is no previous conversation history provided. Therefore, I cannot provide information about what we talked about. I will inform the user that there is no prior conversation to reference. </thinking>

There is no previous conversation history to reference. Please provide specific information or a new question for me to assist you with.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '<thinking> There is no previous conversation history provided. Therefore, I cannot provide information about what we talked about. I will inform the user that there is no prior conversation to reference. </thinking>\n\nThere is no previous conversation history to reference. Please provide specific information or a new question for me to assist you with.'}]}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[0.8608589172363281], traces=[<strands.telemetry.metrics.Trace object at 0xffff42f73f90>], accumulated_usage={'inputTokens': 1832, 'outputTokens': 65, 'totalTokens': 1897}, accumulated_metrics={'latencyMs': 839}), state={})

 You will see the agent forget the past conversation about Hong Kong's weather enquiry. This demonstrates the Strands Agents built-in conversation management is Short-Term Memory only. To enable Long-Term Memory across sessions and agents, you will explore Bedrock AgentCore Memory as a managed agent memory service in [Lab 6: Strands Agents with Bedrock AgentCore Memory](../06-bedrock-agentcore-memory/06-agentcore-memory.ipynb) later.

## Summary

In this introduction to Strands Agents, you've learned:

### What We Accomplished
- **Created your first AI agent** with built-in tools (calculator)
- **Built custom tools** using the `@tool` decorator for specific use cases
- **Explored conversation management** with different ConversationManager types
- **Examined agent execution flow** and message history tracking

### Additional Strands Capabilities
Beyond what we've covered, Strands Agents offers:

- **Model Context Protocol (MCP) Support**: Integrate with MCP servers for extended tool capabilities
- **Multi-Agent Patterns**: Coordinate multiple agents for complex workflows
- **Session Management**: Persistent conversation storage and retrieval
- **Streaming Responses**: Real-time response generation for better user experience
- **Custom Model Integration**: Support for various LLM providers beyond Amazon Bedrock

### Next Steps: Bedrock AgentCore Integration

Now that you understand the fundamentals of Strands Agents, we'll explore how to enhance these capabilities by integrating with **Amazon Bedrock AgentCore**. This integration provides:

- **Code Interpreter**: Execute Python code dynamically within agents
- **Browser Automation**: Web interaction and data extraction capabilities  
- **Secure Credential Management**: Safe handling of external API keys and secrets
- **Runtime Deployment**: Scalable agent deployment in cloud environments
- **Enhanced Observability**: Monitoring and debugging tools for production agents